In [ ]:
import pandas as pd
import numpy as np
import os
import time
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Data Preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder

# Machine Learning Models
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier

# Model Evaluation
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score, f1_score, confusion_matrix

from sklearn.metrics import roc_auc_score, roc_curve, mean_squared_error, r2_score
from sklearn.model_selection import cross_val_score, GridSearchCV
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 100)
from sklearn.metrics import make_scorer

from sklearn.metrics import classification_report, accuracy_score
import random
random.seed(100)

import time
import pyodbc
print(pyodbc.drivers())

### Setting up SQL database

In [ ]:
def create_sql_connection(server, database, username, password, driver='{ODBC Driver 17 for SQL Server}'):
    """
    Establish a connection to a SQL Server database using pyodbc.

    Parameters:
    - server (str): The SQL Server address (e.g., 'localhost' or server IP).
    - database (str): The name of the database you want to connect to.
    - username (str): SQL Server username.
    - password (str): SQL Server password.
    - driver (str): ODBC driver to use. Default is '{ODBC Driver 17 for SQL Server}'.s

    Returns:
    - conn: A pyodbc connection object if successful.
    """
    connection_string = f"""
        DRIVER={driver};
        SERVER={server};
        DATABASE={database};
        UID={username};
        PWD={password};
    """
    try:
        conn = pyodbc.connect(connection_string)
        print("Connection established successfully!")
        return conn
    except Exception as e:
        print(f"Failed to connect to the database. Error: {e}")
        return None

In [ ]:
def query_data(conn, query):
    """
    Execute a SQL query and fetch results as a pandas DataFrame.
    
    Parameters:
    - conn: A pyodbc connection object.
    - query (str): The SQL query to be executed.
    
    Returns:
    - df: A pandas DataFrame containing the query result.
    """
    start_time = time.time()  # Start time measurement
    try:
        cursor = conn.cursor()
        cursor.execute(query)
        
        # Fetch all results from the query
        rows = cursor.fetchall()
        
        # Get column names from cursor
        columns = [desc[0] for desc in cursor.description]
        
        # Create a pandas DataFrame from the results
        df = pd.DataFrame.from_records(rows, columns=columns)
        
    except pyodbc.Error as e:
        print(f"Error executing query: {e}")
        return None
    
    finally:
        cursor.close()
    
    end_time = time.time()  # End time measurement
    execution_time = end_time - start_time  # Calculate execution time
    
    # Print the DataFrame and execution time
    print(f"Query executed in: {execution_time:.4f} seconds")
    
    return df  

In [ ]:
server = 'ROHIT'     
database = 'MedicareClaim'  
username = 'rohit_kosamkar'       
password = 'September@2024' 

In [ ]:
# Establish connection
conn = create_sql_connection(server, database, username, password)

In [ ]:
bene_df  = query_data(conn, "select top 10 * from beneficiarydata")
bene_df.head()

### Set Up MLflow Tracking with SQL Database

In [ ]:
import mlflow
import mlflow.sklearn
from sqlalchemy import create_engine

try:
    # Set MLflow Tracking URI using SQL Server and Windows Authentication
    mlflow.set_tracking_uri("mssql+pyodbc://ROHIT/MedicareClaim?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes")
 
    # Name the experiment
    mlflow.set_experiment("MedicareClaim_Fraud_Detection")
except Exception as e:
    print(f"Connection failed: {e}")

In [ ]:
import mlflow
import mlflow.sklearn
from sqlalchemy import create_engine, text
import pandas as pd

# Use a raw string to handle backslashes in the server name
mlflow_tracking_uri = (
    r"mssql+pyodbc://ROHIT/MedicareClaim"
    "?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes"
)
 
# Set the MLflow Tracking URI
mlflow.set_tracking_uri(mlflow_tracking_uri)
 
try:
    # Create an SQLAlchemy engine
    engine = create_engine(mlflow_tracking_uri)
 
    # Test the connection by running a simple query
    with engine.connect() as connection:
        # Use the text() function to create a SQL statement
        query = text("SELECT TABLE_NAME FROM INFORMATION_SCHEMA.TABLES")
        result = connection.execute(query)
 
        # Fetch all the results and display them
        tables = result.fetchall()
 
        # Print the list of tables
        print("Connection successful! Here are the tables in the database:")
        for table in tables:
            print(table[0])
 
    # Name the experiment (if connection is successful)
    mlflow.set_experiment("MedicareClaim_Fraud_Detection")
 
except Exception as e:
    print(f"Connection failed: {e}")

### Importing data

In [ ]:
data = pd.read_csv(r'../data/interim/model_data_new.csv')
data.shape

In [ ]:
del data['Unnamed: 0']
data.head()

In [ ]:
data.columns

In [ ]:
imp_cols = ['BeneID', 'ClaimID', 'Provider', 'InscClaimAmtReimbursed',
       'DeductibleAmtPaid', 'ClaimPeriod', 'TimeInHptal', 'Diagnosis Count',
       'Procedures Count', 'SamePhysician', 'OPD_Flag', 'PotentialFraud',
        'Gender', 'Race', 'RenalDiseaseIndicator', 'NoOfMonths_PartACov', 'NoOfMonths_PartBCov',
       'ChronicCond_Alzheimer', 'ChronicCond_Heartfailure',
       'ChronicCond_KidneyDisease', 'ChronicCond_Cancer',
       'ChronicCond_ObstrPulmonary', 'ChronicCond_Depression',
       'ChronicCond_Diabetes', 'ChronicCond_IschemicHeart',
       'ChronicCond_Osteoporasis', 'ChronicCond_rheumatoidarthritis',
       'ChronicCond_stroke', 'IPAnnualReimbursementAmt',
       'IPAnnualDeductibleAmt', 'OPAnnualReimbursementAmt',
       'OPAnnualDeductibleAmt',  'Age',
       'ChronicDisease_Count', 'Total_Claims_Per_Bene',
       'Avg_Reimbursement_Per_Bene', 'Age_At_Claim',
       'Multiple_Chronic_Conditions', 'Claim_To_Deductible_Ratio',
       'Total_Annual_Reimbursement', 'Avg_Reimbursement_By_Provider',
       'Provider_Claim_Frequency', 'High_Risk_Provider']

In [ ]:
# data_encoded.drop(columns={'ClaimID', 'Provider'}).corr().to_clipboard()

In [ ]:
data['PotentialFraud'].value_counts()

In [ ]:
data = data[imp_cols]
data.shape

In [ ]:
data.select_dtypes(include='object').columns

In [ ]:
### One-hot encoding
cat_cols = ['SamePhysician', 'OPD_Flag', 'Gender',
       'Race', 'RenalDiseaseIndicator', 'ChronicCond_Alzheimer',
       'ChronicCond_Heartfailure', 'ChronicCond_KidneyDisease',
       'ChronicCond_Cancer', 'ChronicCond_ObstrPulmonary',
       'ChronicCond_Depression', 'ChronicCond_Diabetes',
       'ChronicCond_IschemicHeart', 'ChronicCond_Osteoporasis',
       'ChronicCond_rheumatoidarthritis', 'ChronicCond_stroke']

data_encoded = pd.get_dummies(data, columns = cat_cols,drop_first=True)
data_encoded.shape

In [ ]:
data_encoded.head(2)
del data_encoded['BeneID']

In [ ]:
# del data_encoded['Provider_Claim_Frequency']

In [ ]:
data_encoded['PotentialFraud'].value_counts(normalize=True)

In [ ]:
# X = data_encoded.drop(columns='PotentialFraud')
X = data_encoded[['ClaimID', 'Provider','Total_Claims_Per_Bene',
    'TimeInHptal',
    'Provider_Claim_Frequency',
    'ChronicCond_stroke_Yes',
    'DeductibleAmtPaid',
    'NoOfMonths_PartBCov',
    'NoOfMonths_PartACov',
    'OPD_Flag_Yes',
    'Diagnosis Count',
    'ChronicDisease_Count', 'Age']]
y = data_encoded['PotentialFraud']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size= 0.3, random_state= 42, stratify=y)
print('X_train:', X_train.shape)
print('X_test:', X_test.shape)
print('y_train:', y_train.shape)
print('y_test:', y_test.shape)

### MLFLOW

In [ ]:
# Set the experiment name
experiment_name = "MedicareClaim_Fraud_Detection"
mlflow.set_experiment(experiment_name)  # Set or create the experiment


In [ ]:
# # List of models to evaluate
# models = {
#     "Logistic Regression": LogisticRegression(class_weight='balanced', random_state=42),
#     "Decision Tree Classifier": DecisionTreeClassifier(),
#     "Random Forest Classifier": RandomForestClassifier(random_state=42),
#     "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss')
# }

In [ ]:
# List of models to evaluate
models = {
    "Logistic Regression": LogisticRegression(class_weight='balanced', random_state=42),
    "Decision Tree Classifier": DecisionTreeClassifier(),
    "Random Forest Classifier": RandomForestClassifier(random_state=42),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss')
}

In [ ]:
# Hyperparameter grids for each model
param_grids = {
    "Logistic Regression": {
        'C': [0.01, 0.1, 1, 10, 100],
        'penalty': ['l1', 'l2'],
        'solver': ['liblinear', 'saga']
    },
    "Decision Tree Classifier": {
        'max_depth': [5, 10, 20, None],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4]
    },
    "Random Forest Classifier": {
        'n_estimators': [100, 200, 500],
        'max_depth': [10, 20, 30, None],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4]
    },
    "XGBoost": {
        'n_estimators': [100, 200, 500],
        'max_depth': [3, 5, 7],
        'learning_rate': [0.01, 0.1, 0.2],
        'subsample': [0.6, 0.8, 1.0]
    }
}

In [ ]:
# Custom function to evaluate models with hyperparameter tuning
def evaluate_models(models, param_grids, X_train, X_test, y_train, y_test):
    # Drop non-predictive columns
    X_train_processed = X_train.drop(['ClaimID', 'Provider'], axis=1)
    X_test_processed = X_test.drop(['ClaimID', 'Provider'], axis=1)
    
    # Define scoring metric to optimize (maximize recall)
    scoring = make_scorer(recall_score)

    # Iterate through each model and perform hyperparameter tuning
    for model_name, model in models.items():
        with mlflow.start_run():
            print(f"\nTuning hyperparameters for: {model_name}")

            # Perform GridSearchCV for hyperparameter tuning
            grid_search = GridSearchCV(model, param_grids[model_name], cv=5, scoring=scoring, n_jobs=-1)
            grid_search.fit(X_train_processed, y_train)

            # Get the best model after hyperparameter tuning
            best_model = grid_search.best_estimator_
            best_params = grid_search.best_params_

            # Log best hyperparameters
            mlflow.log_params(best_params)

            # Make predictions with the best model
            y_train_pred = best_model.predict(X_train_processed)
            y_test_pred = best_model.predict(X_test_processed)

            # Calculate metrics
            metrics = {
                'accuracy': {
                    'train': accuracy_score(y_train, y_train_pred),
                    'test': accuracy_score(y_test, y_test_pred)
                },
                'precision': {
                    'train': precision_score(y_train, y_train_pred),
                    'test': precision_score(y_test, y_test_pred)
                },
                'recall': {
                    'train': recall_score(y_train, y_train_pred),
                    'test': recall_score(y_test, y_test_pred)
                },
                'roc_auc': {
                    'train': roc_auc_score(y_train, best_model.predict_proba(X_train_processed)[:, 1]),
                    'test': roc_auc_score(y_test, best_model.predict_proba(X_test_processed)[:, 1])
                }
            }

            # Log metrics with model name included
            for metric_name, metric_values in metrics.items():
                mlflow.log_metric(f"{model_name}_train_{metric_name}", metric_values['train'])
                mlflow.log_metric(f"{model_name}_test_{metric_name}", metric_values['test'])

            # Log feature importance if applicable
            if hasattr(best_model, 'feature_importances_'):
                feature_importances = best_model.feature_importances_
            elif hasattr(best_model, 'coef_'):
                feature_importances = best_model.coef_[0]
            else:
                feature_importances = None

            if feature_importances is not None:
                # Create a DataFrame for better logging
                importance_df = pd.DataFrame({
                    'Feature': X_train_processed.columns,
                    'Importance': feature_importances
                }).sort_values(by='Importance', ascending=False)

                # Log the feature importances as an artifact
                importance_file_path = f"mlruns/{model_name}_feature_importances.csv"
                importance_df.to_csv(importance_file_path, index=False)
                mlflow.log_artifact(importance_file_path)

            # Log the best model
            mlflow.sklearn.log_model(best_model, model_name)

            # End the run
            mlflow.end_run()

In [ ]:
# Call the function with the models, param grids, and training/testing datasets
evaluate_models(models, param_grids, X_train, X_test, y_train, y_test)

In [ ]:
# os.getcwd()

In [ ]:
# def evaluate_models(models, X_train, X_test, y_train, y_test):
#     # Drop non-predictive columns
#     X_train_processed = X_train.drop(['ClaimID', 'Provider'], axis=1)
#     X_test_processed = X_test.drop(['ClaimID', 'Provider'], axis=1)

#     # Iterate through each model
#     for model_name, model in models.items():
#         with mlflow.start_run():
#             # Train the models
#             model.fit(X_train_processed, y_train)

#             # Make predictions
#             y_train_pred = model.predict(X_train_processed)
#             y_test_pred = model.predict(X_test_processed)

#             # Calculate metrics
#             metrics = {
#                 'accuracy': {
#                     'train': accuracy_score(y_train, y_train_pred),
#                     'test': accuracy_score(y_test, y_test_pred)
#                 },
#                 'precision': {
#                     'train': precision_score(y_train, y_train_pred),
#                     'test': precision_score(y_test, y_test_pred)
#                 },
#                 'recall': {
#                     'train': recall_score(y_train, y_train_pred),
#                     'test': recall_score(y_test, y_test_pred)
#                 },
#                 'roc_auc': {
#                     'train': roc_auc_score(y_train, model.predict_proba(X_train_processed)[:, 1]),
#                     'test': roc_auc_score(y_test, model.predict_proba(X_test_processed)[:, 1])
#                 }
#             }

#             # Log metrics with model name included
#             for metric_name, metric_values in metrics.items():
#                 mlflow.log_metric(f"{model_name}_train_{metric_name}", metric_values['train'])
#                 mlflow.log_metric(f"{model_name}_test_{metric_name}", metric_values['test'])

#             # Log feature importance if applicable
#             if hasattr(model, 'feature_importances_'):
#                 feature_importances = model.feature_importances_
#             elif hasattr(model, 'coef_'):
#                 feature_importances = model.coef_[0]
#             else:
#                 feature_importances = None

#             if feature_importances is not None:
#                 # Create a DataFrame for better logging
#                 importance_df = pd.DataFrame({
#                     'Feature': X_train_processed.columns,
#                     'Importance': feature_importances
#                 }).sort_values(by='Importance', ascending=False)

#                 # Log the feature importances as an artifact
#                 importance_file_path = f"mlruns/{model_name}_feature_importances.csv"
#                 importance_df.to_csv(importance_file_path, index=False)
#                 mlflow.log_artifact(importance_file_path)

#             # Log the model
#             mlflow.sklearn.log_model(model, model_name)

#             # End the run
#             mlflow.end_run()

In [ ]:
# # Call the function to evaluate the models
# evaluate_models(models, X_train, X_test, y_train, y_test)

##### Run this in CMD:

mlflow ui --backend-store-uri "mssql+pyodbc://ROHIT/MedicareClaim?driver=ODBC+Driver+17+for+SQL+Server`&trusted_connection=yes"


### Best Model

In [ ]:
# Logistic Regression
start_time = time.time()

# Initialize and fit the Logistic Regression model
# rf_model = RandomForestClassifier(n_estimators=100, max_depth=10, min_samples_split = 7, min_samples_leaf= 2, random_state=42)
lr_model = LogisticRegression(C=0.01, penalty='l2', solver='liblinear')
lr_model.fit(X_train.drop(['ClaimID', 'Provider'], axis=1), y_train)

# Predictions
y_pred_lr_train = lr_model.predict(X_train.drop(['ClaimID', 'Provider'], axis=1))
y_pred_lr_test = lr_model.predict(X_test.drop(['ClaimID', 'Provider'], axis=1))

end_time = time.time()
# Time taken
execution_time = end_time - start_time
print('Execution time: ', execution_time)

# Metrics for Training Data
print("\nLogistic Regression Training Report:")
print(classification_report(y_train, y_pred_lr_train))
print(f"Training Accuracy: {accuracy_score(y_train, y_pred_lr_train)}")
print(f"Training Precision: {precision_score(y_train, y_pred_lr_train)}")
print(f"Training Recall: {recall_score(y_train, y_pred_lr_train)}")
print(f"Training F1 Score: {f1_score(y_train, y_pred_lr_train)}")

# Metrics for Test Data
print("\nLogistic Regression Test Report:")
print(classification_report(y_test, y_pred_lr_test))
print(f"Test Accuracy: {accuracy_score(y_test, y_pred_lr_test)}")
print(f"Test Precision: {precision_score(y_test, y_pred_lr_test)}")
print(f"Test Recall: {recall_score(y_test, y_pred_lr_test)}")
print(f"Test F1 Score: {f1_score(y_test, y_pred_lr_test)}")

In [ ]:
importances = lr_model.coef_[0]
feature_names = X.drop(['ClaimID', 'Provider'], axis=1).columns
feature_importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
feature_importance_df.sort_values(by='Importance', ascending=False, inplace=True)
print(feature_importance_df)

In [ ]:
# feature_importance_df['Feature'][:15].to_list()

In [ ]:
# Step 1: Get probabilities for the positive class (class 1) in the test data
lr_probabilities = lr_model.predict_proba(X_test.drop(['ClaimID', 'Provider'], axis=1))[:, 1]  # Probabilities for class 1


In [ ]:
# del X_test['Fraud_Deciles']

In [ ]:
# Step 2: Add the probabilities to the test dataset
X_test['Fraud_Probabilities'] = lr_probabilities

# Step 3: Create deciles using pd.qcut (10 deciles)
X_test['Fraud_Deciles'] = pd.qcut(X_test['Fraud_Probabilities'].rank(method='first', ascending=False), q=10, labels=range(1, 11))

In [ ]:
X_test.head()

In [ ]:
X_test['Actual_Label'] = y_test.values
X_test['Predicted_Label'] = y_pred_lr_test

X_test['Correctly_Classified'] = (X_test['Actual_Label'] == X_test['Predicted_Label']).astype(int)


In [ ]:
X_test.groupby('Fraud_Deciles').agg(
    total_cases=('Correctly_Classified', 'size'),
    correct_classifications=('Correctly_Classified', 'sum')
)

In [ ]:
X_test[['ClaimID', 'Provider','Fraud_Probabilities', 'Fraud_Deciles', 'Actual_Label',
       'Predicted_Label', 'Correctly_Classified' ]].to_clipboard()

In [ ]:
X_test.columns

### Storing deciles for prediction

In [ ]:
# Step 1: Calculate the min and max probability for each decile
decile_prob_ranges = X_test.groupby('Fraud_Deciles').agg(
    min_probability=('Fraud_Probabilities', 'min'),
    max_probability=('Fraud_Probabilities', 'max')
)

# Display the decile probability ranges
print(decile_prob_ranges)


In [ ]:
decile_prob_ranges

In [ ]:
decile_prob_ranges.to_csv(r'../models/decile_prob_ranges_logistic_regression.csv')

In [ ]:
os.getcwd()

In [ ]:
import pickle

model_filename = '../models/fraud_claim_logistic_model.pkl'

# Save the model to a file in .pkl format
with open(model_filename, 'wb') as file:
    pickle.dump(lr_model, file)

print(f"Model saved as {model_filename}")